# 2 - Deep convolutional GAN


> Part of **[Compressed Sensing using Generative Models](../README.md)**.
> Run `pip install -e .` from the repository root first; every notebook imports
> the `csgm` package rather than redefining the model and recovery code inline.


A GAN reaches the same goal as the VAE - a differentiable map
$G : \mathbb{R}^k \to \mathbb{R}^n$ whose range is the data manifold - by a very
different route. Instead of maximising a likelihood bound, a discriminator $D$
is trained to tell real digits from generated ones while $G$ is trained to fool
it:

$$\min_G \max_D \; \mathbb{E}_{x \sim p_{\text{data}}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))].$$

There is no encoder, so there is no cheap way to ask "which $z$ produced this
image?" - which is precisely the question compressed sensing recovery answers by
optimisation.

In [ ]:
import keras
import matplotlib.pyplot as plt
import numpy as np

from csgm.config import DEFAULT_SEED, MODELS_DIR
from csgm.data import load_mnist
from csgm.models import DCGAN, GANMonitor, build_discriminator, build_generator
from csgm.viz import show_images

keras.utils.set_random_seed(DEFAULT_SEED)

## 2.1 Data

Train and test splits are concatenated. A GAN is a density model that is never
evaluated against labels, so holding 10k digits back would only waste signal.

In [ ]:
(x_train, _), (x_test, _) = load_mnist()
dataset = np.concatenate([x_train, x_test])
print("training on", dataset.shape)

## 2.2 Architecture

Both networks follow the DCGAN recipe (Radford et al., 2015): strided
convolutions instead of pooling, LeakyReLU activations, no fully connected
hidden layers beyond the projection of $z$.

| | generator | discriminator |
|---|---|---|
| input | $z \in \mathbb{R}^k$ | $28 \times 28 \times 1$ image |
| body | dense $\to 3{\times}3{\times}128$, then 3 transposed convolutions (128, 256, 512) | 3 strided convolutions (64, 128, 128) |
| output | $28 \times 28 \times 1$, sigmoid | scalar, sigmoid |

<img src="../docs/figures/dcgan_generator_architecture.png" width="430">
<img src="../docs/figures/dcgan_discriminator_architecture.png" width="430">

In [ ]:
latent_dim = 20

generator = build_generator(latent_dim)
discriminator = build_discriminator()
generator.summary()
discriminator.summary()

## 2.3 Training

Each step updates $D$ on a half-real / half-fake batch, then updates $G$ against
the refreshed $D$. Two details keep the game stable:

- **label smoothing by noise** - discriminator targets are perturbed by
  $\mathcal{U}(0, 0.05)$, which stops $D$ from saturating and starving $G$ of
  gradient;
- **equal learning rates** ($10^{-4}$ for both) - a discriminator that learns
  faster than the generator wins outright and training stalls.

The shipped checkpoints were trained for 50 epochs. Adversarial training has no
convergence criterion to monitor, so `GANMonitor` writes a fixed grid of samples
every epoch and the sheets are inspected by eye.

```bash
python scripts/train_dcgan.py --latent-dim 20 --epochs 50
```

In [ ]:
EPOCHS = 3  # -> 50 to reproduce the checkpoints

gan = DCGAN(discriminator=discriminator, generator=generator, latent_dim=latent_dim)
gan.compile(
    d_optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    g_optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss_fn=keras.losses.BinaryCrossentropy(),
)
history = gan.fit(
    dataset,
    epochs=EPOCHS,
    batch_size=128,
    callbacks=[GANMonitor("../results/samples/notebook_demo", num_images=10)],
)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(history.history["d_loss"], marker="o", ms=3, label="discriminator")
ax.plot(history.history["g_loss"], marker="o", ms=3, label="generator")
ax.set_xlabel("epoch")
ax.set_ylabel("binary cross-entropy")
ax.set_title("Adversarial losses")
ax.grid(alpha=0.3)
ax.legend(frameon=False);

Unlike the VAE curves, these losses are **not** a quality signal: they measure
who is currently winning, not how good the samples are. A generator loss that
falls to zero usually means the discriminator has collapsed, not that the digits
look good. Sample sheets are the real diagnostic.

## 2.4 Samples

In [ ]:
z = np.random.default_rng(0).standard_normal((16, latent_dim)).astype("float32")
show_images(generator(z, training=False), suptitle=f"DCGAN samples after {EPOCHS} epochs");

Three epochs is not enough for clean digits, which is the honest picture of how
slowly adversarial training converges. The shipped checkpoint was trained for
50 epochs; loading it shows what the recovery experiments actually work with.

In [ ]:
from csgm.models import load_generator

trained = load_generator("dcgan", latent_dim)
show_images(trained(z, training=False), suptitle="Shipped checkpoint, 50 epochs, same latent codes");

## 2.5 Why two latent dimensions were trained

$k$ trades expressiveness against searchability. A larger latent space can
represent more digits faithfully, but recovery has to optimise over more
variables from the same number of measurements - and the sample-complexity bound
of Bora et al. grows as $O(k \log L)$. Both $k = 20$ and $k = 30$ were trained so
that trade-off can be measured rather than guessed; see
[notebook 4](04_compressed_sensing_recovery.ipynb).

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
# generator.save(MODELS_DIR / f"gan_gen_dim{latent_dim}.keras")       # uncomment to overwrite
# discriminator.save(MODELS_DIR / f"gan_disc_dim{latent_dim}.keras")
print("checkpoints stay untouched")

---
Next: [3 - Lasso baseline](03_lasso_baseline.ipynb), the sparsity prior the
learned priors are compared against.